# village-crossdate-frames

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from uavloc.data.drone_frames import load_frames
from uavloc.data.satellite import GeoImage
from uavloc.datasets import data_root, verify_dataset, write_manifest
from uavloc.geo import LatLon, footprint_m
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H
from uavloc.track import REPO_ROOT

# Params

In [ ]:
NAME = "village-crossdate-frames"
VERSION = "1"
SOURCE = "ua-village-multidate"
PLACES = {                        # MobileNet / round-3 satellite val: rows the models never trained on (None = whole tile)
    "bushcha-50p2422-25p8305": None,
    "birky-49p9105-23p9192": (0, 704),   # held-out strip: STRIP_ROWS 640 + GAP_PX 64
}
DRONE_DATE = "current"            # newest Esri date: what the drone sees today
FRAMES_PER_PLACE = 20
ALTITUDE_M = 40.0                 # 2025 median AGL 39.5 m
FOV_W_DEG = 60.0                  # correlator geometry
MARGIN_M = 80.0                   # keeps the 1.7x patch + 15 m prior + rotation inside the tile
SEED = 42
DESCRIPTION = ("Simulated drone frames with exact truth: the newest Esri date cropped at random poses on the segmentation "
               "models' satellite val ground (bushcha, birky strip), paired with the oldest date of the same village as the map.")

# Data

In [ ]:
SOURCE_ROOT = REPO_ROOT / "data" / "derived" / SOURCE
out = data_root(NAME, create=True)
rng = np.random.default_rng(SEED)
footprint = footprint_m(ALTITUDE_M, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)
footprint

In [ ]:
def esri_dates(place: str) -> list[str]:
    """Esri Wayback dates on disk, oldest first; `google` is excluded (co-registered to metres, not centimetres)."""
    return sorted(p.stem for p in (SOURCE_ROOT / place).glob("*.jpg") if p.stem not in ("current", "google"))

In [ ]:
def load_date(place: str, date: str) -> GeoImage:
    bounds = json.loads((SOURCE_ROOT / place / "meta.json").read_text())["bounds"]
    return GeoImage.load(SOURCE_ROOT / place / f"{date}.jpg", LatLon(bounds["top"], bounds["left"]),
                         LatLon(bounds["bottom"], bounds["right"]))

In [ ]:
{place: esri_dates(place) for place in PLACES}

# Build

In [ ]:
def random_pose(image: GeoImage, rows: tuple[int, int] | None) -> tuple[LatLon, float]:
    """Uniform position at least MARGIN_M inside the allowed rows (whole tile if None), uniform heading."""
    mpp_x, mpp_y = image.mpp_xy
    first, last = rows or (0, image.pixels.shape[0])
    x_m = rng.uniform(MARGIN_M, image.size_m[0] - MARGIN_M)
    y_m = rng.uniform(first * mpp_y + MARGIN_M, last * mpp_y - MARGIN_M)
    return image.to_latlon(x_m / mpp_x, y_m / mpp_y), float(rng.uniform(0, 360))

In [ ]:
def write_frame(place: str, index: int, drone: GeoImage, map_date: str, rows: tuple[int, int] | None) -> dict:
    """One "drone" frame: the newest date cropped heading-up at a known pose, plus the map it is matched against."""
    position, yaw_deg = random_pose(drone, rows)
    name = f"{place.split('-')[0]}-{index:02d}"
    folder = out / name
    folder.mkdir(exist_ok=True)
    Image.fromarray(drone.crop_rotated(position, yaw_deg, *footprint)).save(folder / "frame.jpg", quality=95)
    bounds = json.loads((SOURCE_ROOT / place / "meta.json").read_text())["bounds"]
    meta = {"place": place, "allowed_rows": list(rows) if rows else None, "lat": position.lat, "lon": position.lon, "alt_m": ALTITUDE_M, "yaw_deg": yaw_deg,
            "mpp": drone.mpp, "drone_date": DRONE_DATE, "map_date": map_date,
            "map_image": str((SOURCE_ROOT / place / f"{map_date}.jpg").relative_to(REPO_ROOT)),
            "map_top_left": [bounds["top"], bounds["left"]], "map_bottom_right": [bounds["bottom"], bounds["right"]]}
    (folder / "meta.json").write_text(json.dumps(meta, indent=1))
    return meta

In [ ]:
metas = []
for place, rows in PLACES.items():
    drone = load_date(place, DRONE_DATE)
    metas += [write_frame(place, i, drone, esri_dates(place)[0], rows) for i in range(FRAMES_PER_PLACE)]
len(metas)

# Checks

In [ ]:
frames = load_frames(NAME)
map_dates = {m["place"]: m["map_date"] for m in metas}
checks = {
    "frames": len(frames),
    "places": list(PLACES),
    "map_dates": map_dates,
    "drone_date": DRONE_DATE,
    "frame_px": list(np.asarray(frames[0].image()).shape[:2]),
    "footprint_m": [round(v, 2) for v in footprint],
}
checks

# Plots

In [ ]:
def preview_row(axes: np.ndarray, meta: dict, frame) -> None:
    """The drone view and the stale map at the same pose, side by side."""
    stale = load_date(meta["place"], meta["map_date"]).crop_rotated(frame.position, frame.yaw_deg, *footprint)
    for ax, image, title in [(axes[0], frame.image(), f"drone view ({DRONE_DATE})"), (axes[1], stale, f"map ({meta['map_date']})")]:
        ax.imshow(image); ax.set_title(title, fontsize=8); ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(frame.name, fontsize=8)

In [ ]:
by_name = {f.name: f for f in frames}
shown = [next(m for m in metas if m["place"] == place) for place in PLACES]
fig, axes = plt.subplots(len(shown), 2, figsize=(8, 2.4 * len(shown)))
for row, meta in zip(axes, shown):
    preview_row(row, meta, by_name[f"{meta['place'].split('-')[0]}-00"])
fig.tight_layout()
fig.savefig(REPO_ROOT / "datasets" / NAME / "preview.png", dpi=90)

# Register

In [ ]:
write_manifest(
    NAME,
    version=VERSION,
    description=DESCRIPTION,
    source=SOURCE,
    config={"places": {k: list(v) if v else None for k, v in PLACES.items()}, "drone_date": DRONE_DATE, "frames_per_place": FRAMES_PER_PLACE, "altitude_m": ALTITUDE_M,
            "fov_w_deg": FOV_W_DEG, "margin_m": MARGIN_M, "seed": SEED},
    samples=len(frames),
    checks=checks,
)
verify_dataset(NAME)